# RAGAS Evaluation Notebook

This notebook reproduces the evaluation results reported in Section 9 of the project report. It includes patches to allow terminal-based scripts (`run_eval.py`) to run safely inside the Jupyter environment without crashing.

In [ ]:
import sys
if len(sys.argv) > 1:
    sys.argv[1] = "jupyter_temp"
else:
    sys.argv.append("jupyter_temp")

# Standard Imports
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
import json

In [ ]:
# Import project modules
from assign_eval_questions import assign_questions, TEAM_MEMBERS
from eval_dataset import eval_questions

import run_eval  # Imported as module to access global variables
from run_eval import (
    run_pipeline_on_test_set,
    score_with_ragas,
    eval_profile
)

from merge_and_score import (
    merge_results,
    print_assignment_breakdown,
    print_category_breakdown,
    print_latency_summary
)

In [ ]:
# Create evaluation dataset
assigned_questions = assign_questions(
    eval_questions,
    TEAM_MEMBERS
)
print(f"Total evaluation questions: {len(assigned_questions)}")

In [ ]:
# Save dataset
with open("eval_dataset_assigned.json","w") as f:
    json.dump(assigned_questions, f, indent=2)

In [ ]:
# Run evaluation
all_results = []

for member in TEAM_MEMBERS:
    print("="*70)
    print(member)
    print("="*70)

    # === BUG FIX 2: OVERRIDE GLOBAL VARIABLE ===
    # Force run_eval to use the current loop's member name 
    # so it saves the JSON file correctly (e.g., eval_results_angel.json)
    run_eval.TEAM_MEMBER = member

    questions = [
        q for q in assigned_questions
        if q["assigned_to"] == member
    ]

    results = run_pipeline_on_test_set(questions)
    all_results.extend(results)

In [ ]:
# Merge results
merged = merge_results()

In [ ]:
# Print dataset statistics
print_assignment_breakdown(merged)
print_category_breakdown(merged)
print_latency_summary(merged)

In [ ]:
# Compute RAGAS
scores = score_with_ragas(merged)

In [ ]:
# Load CSV
df = pd.read_csv("final_scores_combined.csv")
df.head()

In [ ]:
# Overall metrics 
df[
    [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "latency_ms"
    ]
].mean()

In [ ]:
# Category breakdown
df.groupby("category")[
    [
        "faithfulness",
        "answer_relevancy",
        "context_precision"
    ]
].mean()

In [ ]:
# Latency distribution
df["latency_ms"].describe()